In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import scipy.io as sio
import re



In [2]:
def extract_vehicle_type(class_name):
    vehicle_types = ['SUV', 'Sedan', 'Coupe', 'Convertible', 'Hatchback', 
                    'Wagon', 'Van', 'Minivan', 'Cab', 'Pickup']
    
    for vehicle_type in vehicle_types:
        if vehicle_type in class_name:
            return vehicle_type
    
    return "Other"

# Stanford Cars dataset class
class StanfordCarsDataset(Dataset):
    def __init__(self, annotations, class_names, img_dir, transform=None, is_train=True, use_custom_split=True, train_indices=None, val_indices=None):
        """
        Args:
            annotations: Annotations from the MAT file
            class_names: Class names from the MAT file
            img_dir: Directory with all the images
            transform: Optional transform to be applied on a sample
            is_train: Whether this is training set (True) or test set (False)
            use_custom_split: Whether to use custom 80:20 split instead of default split
            train_indices: Indices for training set (only needed if use_custom_split is True)
            val_indices: Indices for validation set (only needed if use_custom_split is True)
        """
        self.annotations = annotations
        self.class_names = class_names
        self.img_dir = img_dir
        self.transform = transform
        self.is_train = is_train
        
        self.filtered_indices = []
        
        if use_custom_split:
            if is_train and train_indices is not None:
                self.filtered_indices = train_indices
            elif not is_train and val_indices is not None:
                self.filtered_indices = val_indices
            else:
                raise ValueError("When use_custom_split is True, train_indices and val_indices must be provided")
        else:
            for i, ann in enumerate(self.annotations):
                if (is_train and ann[-1][0][0] == 0) or (not is_train and ann[-1][0][0] == 1):
                    self.filtered_indices.append(i)
        
        print(f"{'Train' if is_train else 'Validation'} set: {len(self.filtered_indices)} images")
        
        all_vehicle_types = set()
        for class_name_array in class_names[0]:
            vehicle_type = extract_vehicle_type(class_name_array[0])
            all_vehicle_types.add(vehicle_type)
        
        self.vehicle_types = sorted(list(all_vehicle_types))
        self.type_to_idx = {vehicle_type: idx for idx, vehicle_type in enumerate(self.vehicle_types)}
        self.idx_to_type = {idx: vehicle_type for idx, vehicle_type in enumerate(self.vehicle_types)}
        
        print(f"Vehicle types: {self.vehicle_types}")
        print(f"Number of vehicle types: {len(self.vehicle_types)}")
        
    def __len__(self):
        return len(self.filtered_indices)
    
    def __getitem__(self, idx):
        idx = self.filtered_indices[idx]
        ann = self.annotations[idx]
        
        img_path = str(ann[0][0])
        
        x1, y1, x2, y2 = int(ann[1][0][0]), int(ann[2][0][0]), int(ann[3][0][0]), int(ann[4][0][0])
        class_idx = int(ann[5][0][0]) - 1
        class_name = str(self.class_names[0][class_idx][0])
        vehicle_type = extract_vehicle_type(class_name)
        type_idx = self.type_to_idx[vehicle_type]
        
        img_full_path = os.path.join(self.img_dir, img_path)
        image = Image.open(img_full_path).convert('RGB')
        
        image = image.crop((x1, y1, x2, y2))
        
        if self.transform:
            image = self.transform(image)
            
        return image, type_idx


In [5]:
def prepare_data(mat_file, img_dir, use_custom_split=True):
    data = sio.loadmat(mat_file)
    annotations = data['annotations'][0]
    class_names = data['class_names']
    
    train_indices = None
    val_indices = None
    
    if use_custom_split:
        class_to_indices = {}
        for i, ann in enumerate(annotations):
            class_idx = int(ann[5][0][0]) - 1
            if class_idx not in class_to_indices:
                class_to_indices[class_idx] = []
            class_to_indices[class_idx].append(i)
        
        train_indices = []
        val_indices = []
        
        for class_idx, indices in class_to_indices.items():
            np.random.shuffle(indices)
            
            split_idx = int(len(indices) * 0.8)
            
            train_indices.extend(indices[:split_idx])
            val_indices.extend(indices[split_idx:])
        
        np.random.shuffle(train_indices)
        np.random.shuffle(val_indices)
        
        print(f"Custom split created: {len(train_indices)} train samples, {len(val_indices)} validation samples")
    
    train_dataset = StanfordCarsDataset(
        annotations=annotations,
        class_names=class_names,
        img_dir=img_dir,
        transform=data_transforms['train'],
        is_train=True,
        use_custom_split=use_custom_split,
        train_indices=train_indices,
        val_indices=val_indices
    )
    
    val_dataset = StanfordCarsDataset(
        annotations=annotations,
        class_names=class_names,
        img_dir=img_dir,
        transform=data_transforms['val'],
        is_train=False,
        use_custom_split=use_custom_split,
        train_indices=train_indices,
        val_indices=val_indices
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4
    )
    
    return train_loader, val_loader, train_dataset.type_to_idx, train_dataset.idx_to_type

def initialize_model(num_classes):
    model = models.efficientnet_b0(weights='IMAGENET1K_V1')
    
    for param in list(model.parameters())[:-20]:
        param.requires_grad = False
    
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    
    return model

class WarmupPolyLR:
    def __init__(self, optimizer, warmup_epochs, total_epochs, base_lr, min_lr=1e-6, power=0.9):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.power = power
        self.current_epoch = 0
        
    def step(self):
        self.current_epoch += 1
        lr = self._get_lr()
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        return lr
    
    def _get_lr(self):
        if self.current_epoch < self.warmup_epochs:
            alpha = self.current_epoch / self.warmup_epochs
            return self.base_lr * alpha
        else:
            alpha = (self.current_epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            return (self.base_lr - self.min_lr) * (1 - alpha) ** self.power + self.min_lr
            
    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']

def train_model(model, train_loader, val_loader, num_epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    
    scheduler = WarmupPolyLR(
        optimizer, 
        warmup_epochs=WARMUP_EPOCHS, 
        total_epochs=NUM_EPOCHS,
        base_lr=LEARNING_RATE
    )
    
    best_val_acc = 0.0
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for images, labels in train_bar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            current_lr = scheduler.get_lr()
            
            train_bar.set_postfix(loss=loss.item(), acc=correct/total, lr=current_lr)
        
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total
        
        current_lr = scheduler.step()
        
        print(f'Train Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, LR: {current_lr:.6f}')
        
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
            for images, labels in val_bar:
                images, labels = images.to(device), labels.to(device)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
                val_bar.set_postfix(loss=loss.item(), acc=val_correct/val_total)
        
        val_loss = val_loss / len(val_loader.dataset)
        val_acc = val_correct / val_total
        print(f'Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
            }, './type_model/best_model.pth')
            print(f"Model saved with accuracy: {best_val_acc:.4f}")
            
        if (epoch + 1) % 10 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_current_epoch': scheduler.current_epoch,
                'val_acc': val_acc,
                'val_loss': val_loss,
            }, f'./type_model/checkpoint_epoch_{epoch+1}.pth')
            print(f"Checkpoint saved at epoch {epoch+1}")

def evaluate_model(model, test_loader, idx_to_type):
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    pred_types = [idx_to_type[idx] for idx in all_preds]
    true_types = [idx_to_type[idx] for idx in all_labels]
    
    accuracy = sum(p == t for p, t in zip(all_preds, all_labels)) / len(all_preds)
    print(f"Test Accuracy: {accuracy:.4f}")
    
    from sklearn.metrics import confusion_matrix, classification_report
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=list(idx_to_type.values()))
    
    print("Classification Report:")
    print(report)
    
    return pred_types, true_types

In [7]:
mat_file = '/data/ibk5106/dataset/vehicle_datasets/vehicle_info/cars_annos.mat'
img_dir = '/data/ibk5106/dataset/vehicle_datasets/vehicle_info/'

BATCH_SIZE = 32
NUM_EPOCHS = 100
LEARNING_RATE = 0.001
IMAGE_SIZE = 224
WARMUP_EPOCHS = 5
WEIGHT_DECAY = 1e-4

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])  
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


use_custom_split = True
    
train_loader, val_loader, type_to_idx, idx_to_type = prepare_data(mat_file, img_dir, use_custom_split=use_custom_split)

model = initialize_model(num_classes=len(type_to_idx))
model = model.to(device)

train_model(model, train_loader, val_loader, NUM_EPOCHS)

checkpoint = torch.load('./type_model/best_model_type.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1} with validation accuracy: {checkpoint['val_acc']:.4f}")

pred_types, true_types = evaluate_model(model, val_loader, idx_to_type)

for i in range(min(10, len(pred_types))):
    print(f"True: {true_types[i]}, Predicted: {pred_types[i]}")

Using device: cuda
Custom split created: 12873 train samples, 3312 validation samples
Train set: 12873 images
Vehicle types: ['Cab', 'Convertible', 'Coupe', 'Hatchback', 'Minivan', 'Other', 'SUV', 'Sedan', 'Van', 'Wagon']
Number of vehicle types: 10
Validation set: 3312 images
Vehicle types: ['Cab', 'Convertible', 'Coupe', 'Hatchback', 'Minivan', 'Other', 'SUV', 'Sedan', 'Van', 'Wagon']
Number of vehicle types: 10


Epoch 1/100 [Train]: 100%|███████████████████████████████████████████████████████████████████████████| 403/403 [00:22<00:00, 17.84it/s, acc=0.585, loss=1.08, lr=0.001]


Train Loss: 1.1876, Acc: 0.5855, LR: 0.000200


Epoch 1/100 [Val]: 100%|██████████████████████████████████████████████████████████████████████████████████████| 104/104 [00:04<00:00, 21.29it/s, acc=0.709, loss=0.736]


Val Loss: 0.8221, Acc: 0.7086
Model saved with accuracy: 0.7086


...

Train Loss: 0.0063, Acc: 0.9978, LR: 0.000001


Epoch 100/100 [Val]: 100%|████████████████████████████████████████████████████████████████████████████████████| 104/104 [00:04<00:00, 21.30it/s, acc=0.919, loss=0.299]
/tmp/ipykernel_3077894/3035430261.py:47: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Plea

Val Loss: 0.3766, Acc: 0.9191
Checkpoint saved at epoch 100
Loaded best model from epoch 97 with validation accuracy: 0.9233


Evaluating: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 104/104 [00:05<00:00, 19.00it/s]

Test Accuracy: 0.9233
Classification Report:
              precision    recall  f1-score   support

         Cab       0.98      0.98      0.98       308
 Convertible       0.93      0.89      0.91       409
       Coupe       0.87      0.89      0.88       456
   Hatchback       0.93      0.83      0.88       226
     Minivan       0.98      0.95      0.96        85
       Other       0.91      0.85      0.88       229
         SUV       0.97      0.97      0.97       583
       Sedan       0.89      0.95      0.92       779
         Van       0.95      0.99      0.97       120
       Wagon       0.91      0.88      0.90       117

    accuracy                           0.92      3312
   macro avg       0.93      0.92      0.92      3312
weighted avg       0.92      0.92      0.92      3312

True: SUV, Predicted: SUV
True: Hatchback, Predicted: Hatchback
True: Coupe, Predicted: Coupe
True: Convertible, Predicted: Convertible
True: Convertible, Predicted: Convertible
True: Other, Predi